# NUFROST Repeatability Evaluation (Colab)

This notebook assesses the statistical robustness of NUFROST, Zhu2015, and HANTS algorithms by repeating the same evaluation scenarios multiple times with different random seeds.

**Fixed evaluation scenarios:**
- Random‑point masking (40 % of valid observations removed).
- Continuous‑gap simulation (120‑day artificial gap).

**Repeats:** 5 independent runs per image chunk per scenario, each with a different random seed.

The notebook auto‑discovers all (band, lon, lat) cubes, runs the repeats, and saves the raw results. Afterwards, it computes mean and standard deviation of each metric across repeats, both per‑image and aggregated across all images.
Resume‑from‑previous is supported by checking the output CSV.

## 1. Configuration

In [ ]:
from pathlib import Path

MOUNT_POINT_IN_COLAB = Path("/content/drive")
PROJECT_PATH_IN_GDRIVE = Path("WorkSpaces/nufrost")
PROJECT_DIR = MOUNT_POINT_IN_COLAB / "MyDrive" / PROJECT_PATH_IN_GDRIVE
IMAGE_DIR   = PROJECT_DIR / "data/hls"          # or "data/sentinel-2"
OUTPUT_DIR  = PROJECT_DIR / "data/output"
CACHE_DIR   = PROJECT_DIR / "data/cache/colab"
OUTPUT_CSV_PATH = OUTPUT_DIR / "repeatability_results.csv"

# If you want to limit to specific files, list them here; leave empty to auto‑discover.
IMAGE_NAMES = []

# Fixed evaluation settings
RANDOM_POINTS_NUM = 50000          # total random points to mask (for sparse scenario)
CONTINUOUS_GAP_DAYS = 120          # 120‑day artificial gap

# Number of independent repeats per image chunk per scenario
NUM_REPEATS = 5

# Parallel jobs (-1 uses all available cores)
N_JOBS = -1

# Base random seed (each repeat gets a different offset)
BASE_SEED = 42


## 2. Mount Google Drive

In [ ]:
import os
from google.colab import drive # type: ignore[import]

drive.mount(MOUNT_POINT_IN_COLAB.as_posix())
os.chdir(PROJECT_DIR)
print(f"[Working directory changed to: {os.getcwd()}]")


## 3. Install Dependencies

In [ ]:
!apt-get install -y gdal-bin
%pip install -r requirements.txt


## 4. Import Modules

In [ ]:
import src.data_loader
import importlib
import src.evaluation
import pandas as pd
import glob
import re
import numpy as np
import time

from config import build_args
from IPython.display import display

importlib.reload(src.nufrost)
importlib.reload(src.zhu2015)
importlib.reload(src.hants)
importlib.reload(src.evaluation)
importlib.reload(src.data_loader)


## 5. Auto‑discover Image Chunks

In [ ]:
if IMAGE_NAMES:
    image_paths_list = [[(IMAGE_DIR / name).as_posix()] for name in IMAGE_NAMES]
else:
    # Auto‑detect all distinct coordinates and bands, then construct their VRTs
    files = glob.glob((IMAGE_DIR / "*.tif").as_posix())
    loc_ids = set()
    for f in files:
        f_name = Path(f).name
        match = re.search(r"_([A-Z0-9]+)_lon([0-9.]+)_lat([0-9.]+).*?(?:_part\d+)?(?:-\d{10}-\d{10})?\.tif$", f_name)
        if match:
            band = match.group(1)
            lon = float(match.group(2))
            lat = float(match.group(3))
            loc_ids.add((band, lon, lat))

    image_paths_list = []
    for band, lon, lat in loc_ids:
        # find_image_chunks returns the ordered VRT paths for one coordinate/band
        chunks = src.data_loader.find_image_chunks(IMAGE_DIR.as_posix(), lon, lat, band, cache_dir=CACHE_DIR.as_posix())
        if chunks:
            image_paths_list.append(chunks)

print(f"Found {len(image_paths_list)} distinct spatial/band chunks to evaluate.")
if not image_paths_list:
    print("No image chunks found. Please check IMAGE_DIR and filename patterns.")


## 6. Run Repeatability Experiments

In [ ]:
from pathlib import Path

# Ensure output directory exists
Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)

# Load previously evaluated results (if any) to resume
evaluated = set()
if OUTPUT_CSV_PATH.exists():
    try:
        existing_df = pd.read_csv(OUTPUT_CSV_PATH)
        for _, row in existing_df.iterrows():
            evaluated.add((row["Image"], row["Scenario"], row["Repeat"], row["Seed"]))
        print(f"Found existing results for {len(existing_df)} rows. Resuming...")
    except Exception as e:
        print(f"Could not read existing CSV: {e}")

all_results = []

for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    match = re.search(r"([A-Z0-9]+_lon[0-9.]+_lat[0-9.]+)", first_path.stem)
    loc_id = match.group(1) if match else first_path.stem

    print(f"\n--- Evaluating: {loc_id} ---")

    # Two scenarios: random‑point masking and continuous gap
    scenarios = ["random", "gap"]

    for scenario in scenarios:
        for repeat_idx in range(NUM_REPEATS):
            seed = BASE_SEED + hash(loc_id) % 1000 + hash(scenario) % 1000 + repeat_idx
            if (loc_id, scenario, repeat_idx, seed) in evaluated:
                print(f"    Skipping {scenario} repeat {repeat_idx} (already evaluated)")
                continue

            print(f"    Running {scenario} repeat {repeat_idx} (seed={seed})")

            # Build args with default config
            args = build_args({})
            args.image = image_paths
            args.cache_dir = CACHE_DIR.as_posix()
            args.n_jobs = N_JOBS
            args.force_refresh = False

            np.random.seed(seed)

            start_time = time.time()
            if scenario == "random":
                df_results = src.evaluation.evaluate_algorithms(
                    image_path=args.image,
                    args=args,
                    num_points=RANDOM_POINTS_NUM,
                    n_jobs=args.n_jobs,
                )
            else:  # "gap"
                df_results = src.evaluation.evaluate_timeseries_comprehensive(
                    image_path=args.image,
                    args=args,
                    num_samples=RANDOM_POINTS_NUM,
                    simulate_gap_days=CONTINUOUS_GAP_DAYS,
                    n_jobs=args.n_jobs,
                )

            elapsed = time.time() - start_time
            print(f"      finished in {elapsed:.1f}s")

            if df_results.empty:
                print(f"      WARNING: No results for {scenario} repeat {repeat_idx}")
                continue

            # Add metadata columns
            df_results["Image"] = loc_id
            df_results["Scenario"] = scenario
            df_results["Repeat"] = repeat_idx
            df_results["Seed"] = seed

            all_results.append(df_results)

            # Incremental save after each repeat
            Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
            header = not OUTPUT_CSV_PATH.exists()
            df_results.to_csv(OUTPUT_CSV_PATH, mode="a", header=header, index=False)

print("\n========== Repeatability Evaluation Complete ==========")
print(f"Raw results saved to: {OUTPUT_CSV_PATH}")


## 7. Compute Summary Statistics

In [ ]:
if OUTPUT_CSV_PATH.exists():
    df_raw = pd.read_csv(OUTPUT_CSV_PATH)
    print(f"Raw data shape: {df_raw.shape}")
    
    # Group by Image, Scenario, Algorithm and compute mean/std across repeats
    grouped = df_raw.groupby(["Image", "Scenario", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]]
    df_mean = grouped.mean().round(4)
    df_std = grouped.std().round(4)
    
    # Combine mean ± std into a single DataFrame for easy reading
    summary = pd.DataFrame()
    for metric in ["RMSE", "MAE", "R", "OutlierRatio"]:
        summary[f"{metric}_mean"] = df_mean[metric]
        summary[f"{metric}_std"] = df_std[metric]
    
    summary = summary.reset_index()
    
    # Save summary to a separate CSV
    summary_path = OUTPUT_DIR / "repeatability_summary.csv"
    summary.to_csv(summary_path, index=False)
    print(f"Summary statistics saved to: {summary_path}")
    
    # Display overall mean ± std across all images (pooled)
    print("\n--- Overall mean ± std across all images and repeats ---")
    overall_mean = df_raw.groupby(["Scenario", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]].mean().round(4)
    overall_std = df_raw.groupby(["Scenario", "Algorithm"])[["RMSE", "MAE", "R", "OutlierRatio"]].std().round(4)
    
    for scenario in ["random", "gap"]:
        print(f"\nScenario: {scenario}")
        for algo in ["NuFrost", "Zhu2015", "HANTS"]:
            if (scenario, algo) in overall_mean.index:
                mu = overall_mean.loc[(scenario, algo)]
                sigma = overall_std.loc[(scenario, algo)]
                print(f"  {algo}: RMSE {mu['RMSE']}±{sigma['RMSE']}, MAE {mu['MAE']}±{sigma['MAE']}, R {mu['R']}±{sigma['R']}, OutlierRatio {mu['OutlierRatio']}±{sigma['OutlierRatio']}")
    
    # Optionally show a sample of the summary table
    print("\n--- Sample of per‑image summary (first 10 rows) ---")
    display(summary.head(10))
else:
    print("No raw results CSV found.")
